In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

from ADFWI.utils.assessment_metric import MAPE, MSE

base_path = "../multi-parameters/ISO_acoustic/Valhall/data/smooth=8"

init_model = np.load(os.path.join(base_path,"model/init_model.npz"))
true_model = np.load(os.path.join(base_path,"model/true_model.npz"))
init_v   = init_model["vp"]
init_rho = init_model["rho"]
true_v   = true_model["vp"]
true_rho = true_model["rho"]

ox, oz  = 0, 0             # Origin coordinates for x and z directions
nz, nx  = 100, 320         # Grid dimensions in z and x directions
dx, dz  = 25, 25           # Grid spacing in x and z directions
nt, dt  = 3000, 0.0015      # Time steps and time interval
nabc    = 30               # Thickness of the absorbing boundary layer
f0      = 8                # Initial frequency in Hz
free_surface = True        # Enable free surface boundary condition
    
x = np.arange(nx)*dx/1000
z = np.arange(nz)*dz/1000
x_mesh,z_mesh = np.meshgrid(x,z)
src_z = np.array([1  for i in range(4,nx-1,10)])*dz/1000
src_x = np.array([i  for i in range(4,nx-1,10)])*dx/1000
rcv_z = np.array([1  for i in range(0,nx,2)])*dz/1000
rcv_x = np.array([j  for j in range(0,nx,2)])*dz/1000
vmin = true_v.min();vmax = true_v.max()  


In [ ]:
itervp_baseline         = np.load(os.path.join(base_path,"inversion-vp-rho-baseline/iter_vp.npz"))["data"]

itervp_CNN2_2x32      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-2x32/iter_vp.npz"))["data"]
itervp_CNN2_2x64      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-2x64/iter_vp.npz"))["data"]
itervp_CNN2_2x128      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-2x128/iter_vp.npz"))["data"]
itervp_CNN2_3x32      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-3x32/iter_vp.npz"))["data"]
itervp_CNN2_3x64      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-3x64/iter_vp.npz"))["data"]
itervp_CNN2_3x128      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-3x128/iter_vp.npz"))["data"]


iterrho_baseline         = np.load(os.path.join(base_path,"inversion-vp-rho-baseline/iter_rho.npz"))["data"]

iterrho_CNN2_2x32      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-2x32/iter_rho.npz"))["data"]
iterrho_CNN2_2x64      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-2x64/iter_rho.npz"))["data"]
iterrho_CNN2_2x128      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-2x128/iter_rho.npz"))["data"]
iterrho_CNN2_3x32      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-3x32/iter_rho.npz"))["data"]
iterrho_CNN2_3x64      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-3x64/iter_rho.npz"))["data"]
iterrho_CNN2_3x128      = np.load(os.path.join(base_path,"inversion-vp-rho-CNN2-3x128/iter_rho.npz"))["data"]

In [ ]:
def plot_vel_single_for_all(fig,ax,v,title="",MSE="",vmin=None,vmax=None, cmap = "rainbow"):
    plt.rc('font',family='Times New Roman')
    # plm = ax.pcolormesh(x_mesh, z_mesh, v,cmap=cmap,vmin=vmin,vmax=vmax)
    # x = np.arange(nx*3)*dx/3/1000
    # z = np.arange(nz*3)*dz/3/1000
    x = np.arange(nx)*dx/1000
    z = np.arange(nz)*dz/1000
    x_mesh_new,z_mesh_new = np.meshgrid(x,z)
    from scipy.interpolate import griddata
    v_new = griddata((x_mesh.flatten(), z_mesh.flatten()), v.flatten(), (x_mesh_new, z_mesh_new), method='cubic')
    
    plm = ax.pcolormesh(x_mesh_new, z_mesh_new, v_new,cmap=cmap,vmin=vmin,vmax=vmax,shading="nearest")
    ax.invert_yaxis()
    ax.tick_params(labelsize = 16)
    ax.set_title(title,fontsize=16)
    ax.text(0.2,0.4,MSE,fontsize=16,c="k")
    return plm

import matplotlib.transforms as mtransforms
def add_right_cax(ax, pad, width):
    axpos = ax.get_position()
    caxpos = mtransforms.Bbox.from_extents(
        axpos.x1 + pad,
        axpos.y0,
        axpos.x1 + pad + width,
        axpos.y1
    )
    cax = ax.figure.add_axes(caxpos)

    return cax

def add_bottom_cax(ax, pad, height):
    import matplotlib.transforms as mtransforms
    
    axpos = ax.get_position()
    caxpos = mtransforms.Bbox.from_extents(
        axpos.x0,
        axpos.y0 - pad - height,
        axpos.x1,
        axpos.y0 - pad
    )
    cax = ax.figure.add_axes(caxpos)
    
    return cax

def plot_vel_singleline_for_all(ax,v_true,v_init,v_inv,x_distance,title,show_xlabel=True,show_ylabel=False,show_legend=False):
    ax.plot(v_true[:,int(x_distance//dx)]/1000,  z, c='k',   linewidth=2, linestyle="-" ,label="True")
    ax.plot(v_init[:,int(x_distance//dx)]/1000,  z, c='gray',linewidth=2, linestyle="-" ,label="Init")
    ax.plot(v_inv [:,int(x_distance//dx)]/1000,  z, c='r',   linewidth=2, linestyle="--",label="Inverted")
    ax.tick_params(labelsize = 16)
    if not show_xlabel:
        ax.set_xticks([])
    # else:
    #     ax.tick_params(labelsize = 15)
        
    if not show_ylabel:
        ax.set_yticks([])
    else:
        ax.tick_params(labelsize = 12)
    ax.invert_yaxis()
    
    if show_legend:
        ax.legend(fontsize = 12)
    ax.set_title(title,fontsize=16)

## Baseline v.s. One DIP network

In [ ]:

model_mape_vp_baseline   = MAPE(true_v,itervp_baseline[-1])
model_mape_vp_CNN2_2x32  = MAPE(true_v,itervp_CNN2_2x32[-1])
model_mape_vp_CNN2_2x64  = MAPE(true_v,itervp_CNN2_2x64[-1])
model_mape_vp_CNN2_2x128 = MAPE(true_v,itervp_CNN2_2x128[-1])
model_mape_vp_CNN2_3x32  = MAPE(true_v,itervp_CNN2_3x32[-1])
model_mape_vp_CNN2_3x64  = MAPE(true_v,itervp_CNN2_3x64[-1])
model_mape_vp_CNN2_3x128 = MAPE(true_v,itervp_CNN2_3x128[-1])

model_mape_rho_baseline   = MAPE(true_rho,iterrho_baseline[-1])
model_mape_rho_CNN2_2x32  = MAPE(true_rho,iterrho_CNN2_2x32[-1])
model_mape_rho_CNN2_2x64  = MAPE(true_rho,iterrho_CNN2_2x64[-1])
model_mape_rho_CNN2_2x128 = MAPE(true_rho,iterrho_CNN2_2x128[-1])
model_mape_rho_CNN2_3x32  = MAPE(true_rho,iterrho_CNN2_3x32[-1])
model_mape_rho_CNN2_3x64  = MAPE(true_rho,iterrho_CNN2_3x64[-1])
model_mape_rho_CNN2_3x128 = MAPE(true_rho,iterrho_CNN2_3x128[-1])

model_mape_baseline = model_mape_vp_baseline + model_mape_rho_baseline
model_mape_CNN2_2x32 = model_mape_vp_CNN2_2x32 + model_mape_rho_CNN2_2x32
model_mape_CNN2_2x64 = model_mape_vp_CNN2_2x64 + model_mape_rho_CNN2_2x64
model_mape_CNN2_2x128 = model_mape_vp_CNN2_2x128 + model_mape_rho_CNN2_2x128
model_mape_CNN2_3x32 = model_mape_vp_CNN2_3x32 + model_mape_rho_CNN2_3x32
model_mape_CNN2_3x64 = model_mape_vp_CNN2_3x64 + model_mape_rho_CNN2_3x64
model_mape_CNN2_3x128 = model_mape_vp_CNN2_3x128 + model_mape_rho_CNN2_3x128

CNN2_loss_list = np.array([
    model_mape_vp_CNN2_2x32,
    model_mape_vp_CNN2_2x64,
    model_mape_vp_CNN2_2x128,
    model_mape_vp_CNN2_3x32,
    model_mape_vp_CNN2_3x64,
    model_mape_vp_CNN2_3x128,
    
])
np.argmin(CNN2_loss_list)

In [ ]:

fig,axs = plt.subplots(4,2,figsize=(12,11))
im = plot_vel_single_for_all(fig,axs[0][0],true_v    ,title="True Model")
axs[0][0].scatter(rcv_x[1::2],rcv_z[1::2]+0.05,c="w",marker="v",s=10)
axs[0][0].scatter(src_x[1::2],src_z[1::2]+0.05,c="r",marker="*",s=60)
axs[0][0].set_ylabel("Depth (km)",fontsize=15)
axs[0][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[0][1],itervp_baseline[-1]   ,title=r"Baseline"   ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_baseline[-1])))

plot_vel_single_for_all(fig,axs[1][0],itervp_CNN2_2x32[-1]  ,title=r"CNN2-2x32"  ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN2_2x32[-1])))
axs[1][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[1][1],itervp_CNN2_2x64[-1]  ,title=r"CNN2-2x64"  ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN2_2x64[-1])))


plot_vel_single_for_all(fig,axs[2][0],itervp_CNN2_2x128[-1] ,title=r"CNN2-2x128" ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN2_2x128[-1])))
axs[2][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[2][1],itervp_CNN2_3x32[-1]  ,title=r"CNN2-3x32"  ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN2_3x32[-1])))


plot_vel_single_for_all(fig,axs[3][0],itervp_CNN2_3x64[-1] ,title=r"CNN2-3x64"   ,MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN2_3x64[-1])))
axs[3][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[3][1],itervp_CNN2_3x128[-1],title=r"CNN2-3x128",MSE="MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN2_3x128[-1])))

axs[0][0].set_xticks([])
axs[0][1].set_xticks([])
axs[1][0].set_xticks([])
axs[1][1].set_xticks([])
axs[2][0].set_xticks([])
axs[2][1].set_xticks([])
axs[0][1].set_yticks([])
axs[1][1].set_yticks([])
axs[2][1].set_yticks([])
axs[3][1].set_yticks([])

axs[3][0].set_xlabel("Distance (km)", fontsize=15)
axs[3][1].set_xlabel("Distance (km)", fontsize=15)

plt.subplots_adjust(hspace=0.2,wspace=0.1)

cbar = fig.colorbar(im, ax=axs,orientation='vertical',pad = 0.02,shrink=0.6)
cbar.ax.set_title("(m/s)",fontsize=15,loc='center')
cbar.ax.tick_params(labelsize=15)

# plt.savefig(os.path.join(base_path,"DIP_Unet.png"),bbox_inches='tight',dpi=300)
plt.show()

In [ ]:

fig,axs = plt.subplots(4,2,figsize=(12,11))
im = plot_vel_single_for_all(fig,axs[0][0],true_v    ,title="True Model")
axs[0][0].scatter(rcv_x[1::2],rcv_z[1::2]+0.05,c="w",marker="v",s=10)
axs[0][0].scatter(src_x[1::2],src_z[1::2]+0.05,c="r",marker="*",s=60)
axs[0][0].set_ylabel("Depth (km)",fontsize=15)
axs[0][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[0][1],iterrho_baseline[-1]   ,title=r"Baseline"   ,MSE="MAPE:{:.2f}".format(MAPE(true_rho,iterrho_baseline[-1])))

plot_vel_single_for_all(fig,axs[1][0],iterrho_CNN2_2x32[-1]  ,title=r"CNN2-2x32"  ,MSE="MAPE:{:.2f}".format(MAPE(true_rho,iterrho_CNN2_2x32[-1])))
axs[1][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[1][1],iterrho_CNN2_2x64[-1]  ,title=r"CNN2-2x64"  ,MSE="MAPE:{:.2f}".format(MAPE(true_rho,iterrho_CNN2_2x64[-1])))

plot_vel_single_for_all(fig,axs[2][0],iterrho_CNN2_2x128[-1] ,title=r"CNN2-2x128"   ,MSE="MAPE:{:.2f}".format(MAPE(true_rho,iterrho_CNN2_2x128[-1])))
axs[2][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[2][1],iterrho_CNN2_3x32[-1]  ,title=r"CNN2-3x32"    ,MSE="MAPE:{:.2f}".format(MAPE(true_rho,iterrho_CNN2_3x32[-1])))

plot_vel_single_for_all(fig,axs[3][0],iterrho_CNN2_3x64[-1] ,title=r"CNN2-3x64"     ,MSE="MAPE:{:.2f}".format(MAPE(true_rho,iterrho_CNN2_3x64[-1])))
axs[3][0].set_ylabel("Depth (km)",fontsize=15)
plot_vel_single_for_all(fig,axs[3][1],iterrho_CNN2_3x128[-1],title=r"CNN2-3x128"    ,MSE="MAPE:{:.2f}".format(MAPE(true_rho,iterrho_CNN2_3x128[-1])))

axs[0][0].set_xticks([])
axs[0][1].set_xticks([])
axs[1][0].set_xticks([])
axs[1][1].set_xticks([])
axs[2][0].set_xticks([])
axs[2][1].set_xticks([])
axs[0][1].set_yticks([])
axs[1][1].set_yticks([])
axs[2][1].set_yticks([])
axs[3][1].set_yticks([])

axs[3][0].set_xlabel("Distance (km)", fontsize=15)
axs[3][1].set_xlabel("Distance (km)", fontsize=15)

plt.subplots_adjust(hspace=0.2,wspace=0.1)

cbar = fig.colorbar(im, ax=axs,orientation='vertical',pad = 0.02,shrink=0.6)
cbar.ax.set_title("(m/s)",fontsize=15,loc='center')
cbar.ax.tick_params(labelsize=15)

# plt.savefig(os.path.join(base_path,"DIP_Unet.png"),bbox_inches='tight',dpi=300)
plt.show()

## Article Figure

In [ ]:
from matplotlib.ticker import MaxNLocator
plt.rcParams['svg.fonttype'] = 'none'

vpmin, vpmax    = true_v.min(), true_v.max()
rhomin, rhomax  = true_rho.min(), true_rho.max()

iters = -1

fig, axs = plt.subplots(3, 2, figsize=(10, 7))

# Plot for true values
im1 = plot_vel_single_for_all(fig, axs[0][0], true_v, vmin=vpmin, vmax=vpmax, MSE="(a)")
im2 = plot_vel_single_for_all(fig, axs[0][1], true_rho, vmin=rhomin, vmax=rhomax, MSE="(b)")

# Plot for baseline
plot_vel_single_for_all(fig, axs[1][0], itervp_baseline[-1] , MSE="(c) MAPE:{:.2f}".format(MAPE(true_v,itervp_baseline[-1])), vmin=vpmin, vmax=vpmax)
plot_vel_single_for_all(fig, axs[1][1], iterrho_baseline[-1], MSE="(d) MAPE:{:.2f}".format(MAPE(true_rho,iterrho_baseline[-1])), vmin=rhomin, vmax=rhomax)

# Plot for DNN
plot_vel_single_for_all(fig, axs[2][0], itervp_CNN2_2x128[-1], MSE="(e) MAPE:{:.2f}".format(MAPE(true_v,itervp_CNN2_2x128[-1])), vmin=vpmin, vmax=vpmax)
plot_vel_single_for_all(fig, axs[2][1], iterrho_CNN2_2x128[-1], MSE="(f) MAPE:{:.2f}".format(MAPE(true_rho,iterrho_CNN2_2x128[-1])), vmin=rhomin, vmax=rhomax)

axs[0][0].set_ylabel("True", fontsize=15)
axs[1][0].set_ylabel("Baseline", fontsize=15)
axs[2][0].set_ylabel("DNN", fontsize=15)

# Add scatter for RCV and Source points
axs[0][0].scatter(rcv_x[1::2], rcv_z[1::2] + 0.05, c="w", marker="v", s=10,zorder=10)
axs[0][0].scatter(src_x[1::2], src_z[1::2] + 0.05, c="r", marker="*", s=60,zorder=10)

# Set titles for each subplot
axs[0][0].set_title(r"P-wave Velocity", fontsize=15)
axs[0][1].set_title(r"Density", fontsize=15)

# Remove ticks for the upper plots
axs[0][0].set_xticks([])
axs[0][1].set_xticks([])
axs[1][0].set_xticks([])
axs[1][1].set_xticks([])

axs[0][0].set_yticks([])
axs[1][0].set_yticks([])
axs[2][0].set_yticks([])

# Move ticks and labels to the right side for axs[0][1]
axs[0][1].tick_params(axis='y', direction='out', length=2, width=1, colors='black', grid_color='black', grid_alpha=0.5)
axs[0][1].yaxis.set_ticks_position('right')
axs[0][1].set_ylabel("Depth (km)", fontsize=15, labelpad=15,rotation=270)
axs[0][1].yaxis.set_label_position("right")

axs[1][1].tick_params(axis='y', direction='out', length=2, width=1, colors='black', grid_color='black', grid_alpha=0.5)
axs[1][1].yaxis.set_ticks_position('right')
axs[1][1].set_ylabel("Depth (km)", fontsize=15, labelpad=15,rotation=270)
axs[1][1].yaxis.set_label_position("right")

axs[2][1].tick_params(axis='y', direction='out', length=2, width=1, colors='black', grid_color='black', grid_alpha=0.5)
axs[2][1].yaxis.set_ticks_position('right')
axs[2][1].set_ylabel("Depth (km)", fontsize=15, labelpad=15,rotation=270)
axs[2][1].yaxis.set_label_position("right")

# set the ticks number
axs[0][1].yaxis.set_major_locator(MaxNLocator(nbins=4))
axs[1][1].yaxis.set_major_locator(MaxNLocator(nbins=4))
axs[2][1].yaxis.set_major_locator(MaxNLocator(nbins=4))
axs[2][0].xaxis.set_major_locator(MaxNLocator(nbins=6))
axs[2][1].xaxis.set_major_locator(MaxNLocator(nbins=6))

# Set x-labels for the bottom row plots
axs[2][0].set_xlabel("Distance (km)", fontsize=15)
axs[2][1].set_xlabel("Distance (km)", fontsize=15)

# Colorbars on the right side
cax1 = add_bottom_cax(axs[2][0], pad=0.1, height=0.02)
cbar1 = fig.colorbar(im1, cax=cax1, orientation='horizontal')
cbar1.ax.tick_params(labelsize=12)

cax2 = add_bottom_cax(axs[2][1], pad=0.1, height=0.02)
cbar2 = fig.colorbar(im2, cax=cax2, orientation='horizontal')
cbar2.ax.tick_params(labelsize=12)

# Add legends to the right side of the subplots
axs[0][0].legend(["Receiver", "Source"], loc='center left', fontsize=12, bbox_to_anchor=(1, 0.5), frameon=False)

# Adjust space between subplots
plt.subplots_adjust(hspace=0.1, wspace=0.05)

# plt.savefig("./Figures/Figure_S6_multiparameters_vp_rho_Valhall.png",bbox_inches='tight',dpi=300)
# plt.savefig("./Figures_SVG/FigureS6_multiparameters_vp_rho_Valhall.svg",bbox_inches='tight',format="svg")
plt.savefig("./Figures_PDF/FigureS6_multiparameters_vp_rho_Valhall.pdf",bbox_inches='tight',format="pdf",dpi=300)
plt.show()